In [1]:
import pandas as pd
import numpy as np

In [2]:
price = pd.read_csv('ames_prices.csv')

In [3]:
price.head()

,Order,PID,area,price,MS.SubClass,MS.Zoning,Lot.Frontage,Lot.Area,Street,Alley,...,Screen.Porch,Pool.Area,Pool.QC,Fence,Misc.Feature,Misc.Val,Mo.Sold,Yr.Sold,Sale.Type,Sale.Condition
0,1,526301100,1656,215000,20,RL,141.0,31770,Pave,NaN,...,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal
1,2,526350040,896,105000,20,RH,80.0,11622,Pave,NaN,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
2,3,526351010,1329,172000,20,RL,81.0,14267,Pave,NaN,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
3,4,526353030,2110,244000,20,RL,93.0,11160,Pave,NaN,...,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal
4,5,527105010,1629,189900,60,RL,74.0,13830,Pave,NaN,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal


In [ ]:
##7a-7c
from scipy.stats import norm

price = pd.read_csv("ames_prices.csv")


price["central_air"] = (price["Central.Air"] == "Y").astype(int)


p_hat = price["central_air"].mean()

# b. Bootstrap
seed = 200
B = 10_000
rng = np.random.default_rng(seed)

bootstrap_estimates = [
    rng.choice(price["central_air"], size=len(price), replace=True).mean()
    for _ in range(B)
]


bootstrap_se = np.std(bootstrap_estimates, ddof=1)


for level in [0.90, 0.95, 0.99]:
    z_star = norm.ppf(1 - (1 - level) / 2)
    lower = p_hat - z_star * bootstrap_se
    upper = p_hat + z_star * bootstrap_se
    print(f"{level:.0%} CI: ({lower:.6f}, {upper:.6f})")

print(f"Proportion: {p_hat:.6f}")
print(f"Bootstrap SE: {bootstrap_se:.6f}")
print(f"Seed: {seed}, replications: {B}")

90% CI: (0.925484, 0.940728)
95% CI: (0.924024, 0.942188)
99% CI: (0.921170, 0.945042)
Proportion: 0.933106
Bootstrap SE: 0.004634
Seed: 200, replications: 10000


7d) at alpha = 0.1 we reject the null
    at alpha = 0.05 we reject the null
    at alpha = 0.01 we reject the null.
    We can say with 99 percent confidence that the population proportion of homes with central air does not equal 90 percent based on the data in our sample.

In [7]:
##8a-8c
area_asinh = np.arcsinh(price["area"])
price_asinh = np.arcsinh(price["price"])


r_hat = np.corrcoef(area_asinh, price_asinh)[0, 1]


seed = 200
B = 10_000
rng = np.random.default_rng(seed)

n = len(price)
bootstrap_correlations = np.empty(B)

for b in range(B):
    
    indices = rng.integers(0, n, size=n)

    bootstrap_correlations[b] = np.corrcoef(
        area_asinh.iloc[indices],
        price_asinh.iloc[indices]
    )[0, 1]


bootstrap_se = bootstrap_correlations.std(ddof=1)


confidence_intervals = {}

for level in [0.90, 0.95, 0.99]:
    z_star = norm.ppf(1 - (1 - level) / 2)
    margin = z_star * bootstrap_se

    confidence_intervals[level] = (
        r_hat - margin,
        r_hat + margin
    )

print(f"Original correlation: {r_hat:.6f}")
print(f"Seed: {seed}")
print(f"Bootstrap replications: {B:,}")
print(f"Bootstrap standard error: {bootstrap_se:.6f}")

for level, interval in confidence_intervals.items():
    print(
        f"{level:.0%} centered-Normal CI: "
        f"({interval[0]:.6f}, {interval[1]:.6f})"
    )

Original correlation: 0.723166
Seed: 200
Bootstrap replications: 10,000
Bootstrap standard error: 0.009895
90% centered-Normal CI: (0.706891, 0.739441)
95% centered-Normal CI: (0.703773, 0.742559)
99% centered-Normal CI: (0.697679, 0.748653)


8d) at alpha = 0.1 we reject the null
    at alpha = 0.05 we reject the null
    at alpha = 0.01 we reject the null.
    We can say with 99 percent confidence that the population correlation between the arcsinh of area and the arcsinh of price is not 0 based on the data in our sample.

8e) Area and price are very correlated since area is a main predictor of price. These variables have multi colinearity.  If we are looking at predicting central air, expensive homes with large square footage are likely to have central air but both variables will capture this prediction, if we bootstrap samples then the area and the price could end up negating the effects of each other. That is why we must resample them together.

In [ ]:

##9a-9c
metabric = pd.read_csv("metabric.csv")


survival_months = (
    metabric["Overall Survival (Months)"]
    .dropna()
    .to_numpy()
)


mean_months = survival_months.mean()


seed = 200
B = 10_000
rng = np.random.default_rng(seed)

bootstrap_means = np.empty(B)

for b in range(B):
    bootstrap_sample = rng.choice(
        survival_months,
        size=len(survival_months),
        replace=True
    )
    bootstrap_means[b] = bootstrap_sample.mean()

bootstrap_se = bootstrap_means.std(ddof=1)


print(f"Observations: {len(survival_months):,}")
print(f"Estimated mean: {mean_months:.3f} months")
print(f"Seed: {seed}")
print(f"Bootstrap replications: {B:,}")
print(f"Bootstrap SE: {bootstrap_se:.3f} months")

for level in [0.90, 0.95, 0.99]:
    z_star = norm.ppf(1 - (1 - level) / 2)
    lower = mean_months - z_star * bootstrap_se
    upper = mean_months + z_star * bootstrap_se

    print(f"{level:.0%} CI: ({lower:.3f}, {upper:.3f}) months")

Observations: 1,343
Estimated mean: 129.094 months
Seed: 200
Bootstrap replications: 10,000
Bootstrap SE: 2.116 months
90% CI: (125.614, 132.575) months
95% CI: (124.947, 133.242) months
99% CI: (123.644, 134.545) months


9d)  at alpha = 0.1 we reject the null
    at alpha = 0.05 we reject the null
    at alpha = 0.01 we reject the null.
    We can say with 99 percent confidence that the population mean time to death is not equal to 120 based on the data in our sample. 
9e) Our estimate is 10 years and 9 months with a CI interval of 10 years and 5 months to 11 years and three months.  If the patients are currently living than we do not know when they will die. This makes the results not representative of unrestricted mean time. There months in life after visit will just be represented in months since the visit was recorded.

In [ ]:
## 10a-10c

nhanes = pd.read_csv("nhanes_data_17_18.csv")

sleep_col = "SleepHoursWeekdaysOrWorkdays"

age_candidates = [
    "Age",
    "AgeInYears",
    "AgeInYearsAtScreening",
    "RIDAGEYR"
]

age_col = next(
    (col for col in age_candidates if col in nhanes.columns),
    None
)

if age_col is None:
    raise KeyError(
        "Age column not identified. Available columns:\n"
        + "\n".join(nhanes.columns)
    )


age = pd.to_numeric(nhanes[age_col], errors="coerce")
sleep = pd.to_numeric(nhanes[sleep_col], errors="coerce")

adult_sleep = sleep[(age >= 18) & sleep.notna()].to_numpy()


mean_sleep = adult_sleep.mean()

# a. Bootstrap
seed = 200
B = 10_000
rng = np.random.default_rng(seed)

bootstrap_means = np.empty(B)

for b in range(B):
    bootstrap_sample = rng.choice(
        adult_sleep,
        size=len(adult_sleep),
        replace=True
    )
    bootstrap_means[b] = bootstrap_sample.mean()

bootstrap_se = bootstrap_means.std(ddof=1)

print(f"Adult respondents analyzed: {len(adult_sleep):,}")
print(f"Mean weekday sleep: {mean_sleep:.4f} hours")
print(f"Seed: {seed}")
print(f"Bootstrap replications: {B:,}")
print(f"Bootstrap SE: {bootstrap_se:.4f} hours")

for level in [0.90, 0.95, 0.99]:
    z_star = norm.ppf(1 - (1 - level) / 2)
    lower = mean_sleep - z_star * bootstrap_se
    upper = mean_sleep + z_star * bootstrap_se

    print(f"{level:.0%} CI: ({lower:.4f}, {upper:.4f}) hours")

/var/folders/vx/_s5sm_dd32jbj6pctt_5k0xw0000gn/T/ipykernel_63845/3554758487.py:1: DtypeWarning: Columns (0: ServedActiveDutyInUsArmedForces) have mixed types. Specify dtype option on import or set low_memory=False.
  nhanes = pd.read_csv("nhanes_data_17_18.csv")


Adult respondents analyzed: 5,489
Mean weekday sleep: 7.6212 hours
Seed: 200
Bootstrap replications: 10,000
Bootstrap SE: 0.0226 hours
90% CI: (7.5842, 7.6583) hours
95% CI: (7.5770, 7.6654) hours
99% CI: (7.5632, 7.6793) hours


10d)  at alpha = 0.1 we reject the null
    at alpha = 0.05 we reject the null
    at alpha = 0.01 we reject the null.
    We can say with 99 percent confidence that the population mean weekday sleep is not equal to 8 hours based on the data in our sample.

10e) During the weekday the population mean sleep time is statistically less than 8 hours. This is also practically significant because at 95% confidence the difference of at least 20 minutes is still a noticeable difference.